In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import pandas as pd

def compute_cell_type_adjacency(adata_ct, threshold=0.3):
    """
    计算细胞类型之间的连接关系。
    
    参数：
        adata_ct (AnnData): 经过解卷积的 AnnData 对象。
        threshold (float): 细胞类型占比的阈值，默认值为 0.3。
    
    返回：
        pd.DataFrame: 细胞类型邻接矩阵。
    """
    # 计算空间邻居
    sq.gr.spatial_neighbors(adata_ct)
    adjacency_matrix = adata_ct.obsp['spatial_distances']
    
    # 在 adata_ct.obs 中添加与细胞类型匹配的列，并初始化为 "N"
    for cell_type in adata_ct.var_names:
        adata_ct.obs[cell_type + "_dom"] = "N"
    
    # 计算每个 spot 中每种细胞类型的占比
    cell_type_counts = adata_ct.X
    spot_sums = cell_type_counts.sum(axis=1)
    spot_sums[spot_sums == 0] = 1  # 避免除零错误
    cell_type_ratios = cell_type_counts / spot_sums[:, None]
    
    # 根据阈值更新 obs
    for i, cell_type in enumerate(adata_ct.var_names):
        adata_ct.obs[cell_type + "_dom"] = np.where(cell_type_ratios[:, i] > threshold, "Y", "N")
    
    # 计算细胞类型之间的连接关系
    num_cell_types = len(adata_ct.var_names)
    cell_type_adj_matrix = np.zeros((num_cell_types, num_cell_types), dtype=int)
    processed_pairs = set()
    
    for i, spot in enumerate(adata_ct.obs_names):
        present_types = [cell_type for cell_type in adata_ct.var_names if adata_ct.obs[cell_type + "_dom"][i] == "Y"]
        neighbors = adjacency_matrix[i].nonzero()[1]
        
        for neighbor in neighbors:
            if (i, neighbor) in processed_pairs or (neighbor, i) in processed_pairs:
                continue
            processed_pairs.add((i, neighbor))
            
            neighbor_types = [cell_type for cell_type in adata_ct.var_names if adata_ct.obs[cell_type + "_dom"][neighbor] == "Y"]
            
            for type1 in present_types:
                for type2 in neighbor_types:
                    idx1, idx2 = adata_ct.var_names.get_loc(type1), adata_ct.var_names.get_loc(type2)
                    cell_type_adj_matrix[idx1, idx2] += 1
                    cell_type_adj_matrix[idx2, idx1] += 1  # 保持对称性
    
    return pd.DataFrame(cell_type_adj_matrix, index=adata_ct.var_names, columns=adata_ct.var_names)



def generate_hex_grid(n_spots, start_x=11672, start_y=3207, spacing=100):
    """
    生成符合 10X Visium 规则的六边形网格，并随机排列坐标。
    :param n_spots: 总 spot 数量
    :param start_x: 起始 X 坐标
    :param start_y: 起始 Y 坐标
    :param spacing: 邻近 spot 之间的间距
    :return: 随机排列后的坐标数组
    """
    
    # 估算网格尺寸，接近正方形
    approx_side = int(np.sqrt(n_spots))
    n_rows = approx_side
    n_cols = int(n_spots / n_rows) + 1
    
    coords = []
    for row in range(n_rows):
        for col in range(n_cols):
            if len(coords) >= n_spots:
                break
            
            x = start_x + col * spacing
            y = start_y + row * int(spacing * np.sqrt(3)/2)
            
            # 偶数行偏移
            if row % 2 == 1:
                x += spacing // 2
            
            coords.append((x, y))
    
    # 随机排列坐标
    np.random.shuffle(coords)
    
    return np.array(coords)

/home/test/anaconda3/envs/scrna/lib/python3.11/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.3-CAPI-1.17.3) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/home/test/anaconda3/envs/scrna/lib/python3.11/site-packages/spatialdata/__init__.py:11: UserWarning: Geopandas was set to use PyGEOS, changing to shapely 2.0 with:

	geopandas.options.use_pygeos = True

If you intended to use PyGEOS, set the option to False.
  _check_geopandas_using_shapely()


In [ ]:
import itertools
cell_types_comb = ['AC_like','G_cycling', 'MES_Core', 'MES_Hypo', 'NPC_like', 'OPC_like']
combinations = set()
for comb in itertools.combinations(cell_types_comb, 2):
    # 保证组合是对称的
    comb_str = f"{comb[0]}|{comb[1]}"
    combinations.add(comb_str)
    comb_str = f"{comb[0]}|{comb[0]}"
    combinations.add(comb_str)
    comb_str = f"{comb[1]}|{comb[1]}"
    combinations.add(comb_str)

combinations = list(combinations)
combinations.sort()
merged_df = pd.DataFrame(index=combinations)

In [ ]:
n = 50
final_random_df = None  # 先初始化为空

adata_cp_cnv_backup = adata_cp_cnv.copy()
sc.pp.filter_cells(adata_cp_cnv_backup,min_counts = 200)

cell_types = ['AC_like','G_cycling', 'MES', 'MES_Ast', 'MES_Hypo', 'NPC_like', 'OPC_like']
cell_matrix = adata_cp_cnv_backup.obs[cell_types]
cell_matrix['MES_Core'] = cell_matrix['MES'] + cell_matrix['MES_Ast']
cell_matrix = cell_matrix.drop(['MES','MES_Ast'], axis=1)
adata_cp_cnv_random = ann.AnnData(cell_matrix)
adata_cp_cnv_random.obs = adata_cp_cnv_backup.obs
adata_cp_cnv_random.uns = adata_cp_cnv_backup.uns
adata_cp_cnv_random.obsm = adata_cp_cnv_backup.obsm

del adata_cp_cnv_backup

adata_cp_cnv_random.obsm['spatial'] = generate_hex_grid(adata_cp_cnv_random.shape[0])

for _ in range(n):
    np.random.shuffle(adata_cp_cnv_random.obsm['spatial'])  # 对空间信息进行随机打乱
    random_df = compute_cell_type_adjacency(adata_cp_cnv_random, 0.2)  # 计算邻接矩阵

    if final_random_df is None:
        final_random_df = random_df.copy()  # 直接赋值第一次的结果
    else:
        final_random_df += random_df  # 累加

final_random_df = final_random_df / n  # 计算平均值
final_random_df = final_random_df + 1

In [ ]:
from cell2location.utils import select_slide
df_ls = []

unique_values = final_random_df.stack().unique()
all_num = unique_values.sum()


for sample in adata_cp_cnv.obs['sample'].unique().tolist():
    cell_types = ['AC_like','G_cycling', 'MES', 'MES_Ast', 'MES_Hypo', 'NPC_like', 'OPC_like']
    slide = select_slide(adata_cp_cnv, sample)
    sc.pp.filter_cells(slide,min_counts = 200)
    cell_matrix = slide.obs[cell_types]
    cell_matrix['MES_Core'] = cell_matrix['MES'] + cell_matrix['MES_Ast']
    cell_matrix = cell_matrix.drop(['MES','MES_Ast'], axis=1)
    adata_ct = ann.AnnData(cell_matrix)
    adata_ct.obs = slide.obs
    adata_ct.uns = slide.uns
    adata_ct.obsm = slide.obsm
    adata_ct_random = adata_ct.copy()
    org_df = compute_cell_type_adjacency(adata_ct,0.2)
    df_ls.append(org_df)

    unique_values_org = org_df.stack().unique()
    sample_num = unique_values_org.sum()

    org_df = org_df +1
    final_df = (org_df / final_random_df) * (all_num / sample_num)
    print(all_num / sample_num)

    sample_data = []
    for comb in combinations:
        cell_type_1, cell_type_2 = comb.split('|')
        # 取出对应的浮动值
        value = final_df.loc[cell_type_1, cell_type_2]  # 数据框的值
        sample_data.append(value)
    
    merged_df[sample] = sample_data